<a href="https://colab.research.google.com/github/vkjadon/hugging_face/blob/main/content_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q huggingface_hub pandas

In [8]:
from huggingface_hub import HfApi
import inspect

print(inspect.signature(HfApi.list_models))

(self, *, filter: 'str | Iterable[str] | None' = None, author: 'str | None' = None, apps: 'str | list[str] | None' = None, gated: 'bool | None' = None, inference: "Literal['warm'] | None" = None, inference_provider: "Literal['all'] | PROVIDER_T | list[PROVIDER_T] | None" = None, model_name: 'str | None' = None, trained_dataset: 'str | list[str] | None' = None, search: 'str | None' = None, pipeline_tag: 'str | None' = None, num_parameters: 'str | None' = None, emissions_thresholds: 'tuple[float, float] | None' = None, sort: 'ModelSort_T | None' = None, limit: 'int | None' = None, expand: 'list[ExpandModelProperty_T] | None' = None, full: 'bool | None' = None, cardData: 'bool' = False, fetch_config: 'bool' = False, token: 'bool | str | None' = None) -> 'Iterable[ModelInfo]'


In [ ]:
from huggingface_hub import HfApi
import pandas as pd

api = HfApi()

tasks = {
    "Sentiment Analysis": "text-classification",
    "Image Classification": "image-classification",
    "Speech Recognition": "automatic-speech-recognition",
    "Image Captioning": "image-to-text"
}

all_models = []

for task_name, pipeline_tag in tasks.items():

    print(f"\nSearching {task_name} models...")

    models = api.list_models(
        pipeline_tag=pipeline_tag,
        sort="downloads",
        search="imdb",
        limit=10,
        full=True
    )

    for model in models:

        all_models.append({
            "Task": task_name,
            "Model": model.id,
            "Downloads": model.downloads,
            "Likes": model.likes,
            "Pipeline": model.pipeline_tag,
        })

df = pd.DataFrame(all_models)

df.head()

In [ ]:
display(df)

In [3]:
from huggingface_hub import HfApi
import pandas as pd

api = HfApi()

def find_models(
    pipeline_tag=None,
    search_text=None,
    dataset=None,
    limit=10,
    sort_by="downloads"
    ):
    """
    Search Hugging Face models.

    Args:
        pipeline_tag (str): Task name
        search_text (str): Text to search in model names/metadata
        dataset (str): Training dataset name
        limit (int): Number of models to return
        sort_by (str): downloads, likes, created_at, etc.

    Returns:
        DataFrame
    """

    models = api.list_models(
        pipeline_tag=pipeline_tag,
        search=search_text,
        trained_dataset=dataset,
        sort=sort_by,
        limit=limit,
        full=True
    )

    results = []

    for model in models:
        results.append({
            "Model": model.id,
            "Downloads": model.downloads,
            "Likes": model.likes,
            "Task": model.pipeline_tag
        })

    return pd.DataFrame(results)

In [4]:
df = find_models(
    pipeline_tag="text-classification",
    limit=5
)

print(df.to_string(index=False))

                                                     Model  Downloads  Likes                Task
                                   BAAI/bge-reranker-v2-m3   13338750   1007 text-classification
                                          ProsusAI/finbert    6823867   1164 text-classification
                                    BAAI/bge-reranker-base    4556146    236 text-classification
distilbert/distilbert-base-uncased-finetuned-sst-2-english    3407162    900 text-classification
          cardiffnlp/twitter-roberta-base-sentiment-latest    3275637    804 text-classification


In [5]:
tasks = [
    "text-classification",
    "image-classification",
    "automatic-speech-recognition",
    "image-to-text"
]

for task in tasks:
    print(f"\n{'='*80}")
    print(task)
    print(f"{'='*80}")

    df = find_models(
        pipeline_tag=task,
        limit=5
    )

    print(df.to_string(index=False))
    display(df)


text-classification
                                                     Model  Downloads  Likes                Task
                                   BAAI/bge-reranker-v2-m3   13338750   1007 text-classification
                                          ProsusAI/finbert    6823867   1164 text-classification
                                    BAAI/bge-reranker-base    4556146    236 text-classification
distilbert/distilbert-base-uncased-finetuned-sst-2-english    3407162    900 text-classification
          cardiffnlp/twitter-roberta-base-sentiment-latest    3275637    804 text-classification


,Model,Downloads,Likes,Task
0,BAAI/bge-reranker-v2-m3,13338750,1007,text-classification
1,ProsusAI/finbert,6823867,1164,text-classification
2,BAAI/bge-reranker-base,4556146,236,text-classification
3,distilbert/distilbert-base-uncased-finetuned-s...,3407162,900,text-classification
4,cardiffnlp/twitter-roberta-base-sentiment-latest,3275637,804,text-classification



image-classification
                                   Model  Downloads  Likes                 Task
    timm/mobilenetv3_small_100.lamb_in1k   10801148     75 image-classification
          Falconsai/nsfw_image_detection    8527993   1083 image-classification
             google/vit-base-patch16-224    4864809    965 image-classification
    dima806/fairface_age_image_detection    4779067     73 image-classification
timm/convnextv2_nano.fcmae_ft_in22k_in1k    4503157      4 image-classification


,Model,Downloads,Likes,Task
0,timm/mobilenetv3_small_100.lamb_in1k,10801148,75,image-classification
1,Falconsai/nsfw_image_detection,8527993,1083,image-classification
2,google/vit-base-patch16-224,4864809,965,image-classification
3,dima806/fairface_age_image_detection,4779067,73,image-classification
4,timm/convnextv2_nano.fcmae_ft_in22k_in1k,4503157,4,image-classification



automatic-speech-recognition
                                        Model  Downloads  Likes                         Task
                  argmaxinc/whisperkit-coreml    9901494    184 automatic-speech-recognition
             pyannote/speaker-diarization-3.1    9818666   2060 automatic-speech-recognition
                openai/whisper-large-v3-turbo    8054465   3041 automatic-speech-recognition
                      openai/whisper-large-v3    5341634   5752 automatic-speech-recognition
jonatasgrosman/wav2vec2-large-xlsr-53-russian    3501144     75 automatic-speech-recognition


,Model,Downloads,Likes,Task
0,argmaxinc/whisperkit-coreml,9901494,184,automatic-speech-recognition
1,pyannote/speaker-diarization-3.1,9818666,2060,automatic-speech-recognition
2,openai/whisper-large-v3-turbo,8054465,3041,automatic-speech-recognition
3,openai/whisper-large-v3,5341634,5752,automatic-speech-recognition
4,jonatasgrosman/wav2vec2-large-xlsr-53-russian,3501144,75,automatic-speech-recognition



image-to-text
                                 Model  Downloads  Likes          Task
 Salesforce/blip-image-captioning-base    2487691    857 image-to-text
Salesforce/blip-image-captioning-large     730662   1474 image-to-text
      PaddlePaddle/PP-OCRv5_server_det     617363     66 image-to-text
                    PaddlePaddle/UVDoc     496749      9 image-to-text
    PaddlePaddle/PP-LCNet_x1_0_doc_ori     424559     15 image-to-text


,Model,Downloads,Likes,Task
0,Salesforce/blip-image-captioning-base,2487691,857,image-to-text
1,Salesforce/blip-image-captioning-large,730662,1474,image-to-text
2,PaddlePaddle/PP-OCRv5_server_det,617363,66,image-to-text
3,PaddlePaddle/UVDoc,496749,9,image-to-text
4,PaddlePaddle/PP-LCNet_x1_0_doc_ori,424559,15,image-to-text


In [12]:
def analyze_model_card(model_id):
    """Extract key information from model card."""
    info = api.model_info(model_id)
    return {
        "model_id": model_id,
        "downloads": info.downloads,
        "likes": info.likes,
        "license": info.card_data.license if info.card_data else "Unknown",
        "pipeline_tag": info.pipeline_tag,
        "tags": info.tags[:5] if info.tags else [],
    }

# Compare candidates
text_analysis = analyze_model_card("distilbert-base-uncased-finetuned-sst-2-english")
print(f"Text Model: {text_analysis}")

Text Model: {'model_id': 'distilbert-base-uncased-finetuned-sst-2-english', 'downloads': 3407162, 'likes': 900, 'license': 'apache-2.0', 'pipeline_tag': 'text-classification', 'tags': ['transformers', 'pytorch', 'tf', 'rust', 'onnx']}


In [11]:
print(inspect.signature(api.model_info))

(repo_id: 'str', *, revision: 'str | None' = None, timeout: 'float | None' = None, securityStatus: 'bool | None' = None, files_metadata: 'bool' = False, expand: 'list[ExpandModelProperty_T] | None' = None, token: 'bool | str | None' = None) -> 'ModelInfo'
